In [21]:
# Cell 1: Install all required packages
!pip install rank-bm25 sentence-transformers transformers groq google-generativeai torch -q

In [ ]:
# Cell 2: Configure API Keys
import os
from google.colab import userdata  # Remove this line if not using Colab secrets

# Option A: Use Colab Secrets (recommended)
# Go to 🔑 Secrets tab on the left, add GEMINI_API_KEY and GROQ_API_KEY
try:
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
    GROQ_API_KEY   = userdata.get("GROQ_API_KEY")
except:
    # Option B: Paste keys directly (less secure)
    GEMINI_API_KEY = ""
    GROQ_API_KEY   = ""

os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
os.environ["GROQ_API_KEY"]   = GROQ_API_KEY

print("✅ API keys loaded.")

✅ API keys loaded.


In [23]:
# Cell 3: Document Corpus — 12 AI/ML documents
# Requirements met:
#   ✅ 10+ documents
#   ✅ 3+ documents on related but distinct sub-topics (neural network training)
#   ✅ At least 1 with technical jargon BM25 would catch well (BLEU, RLHF, etc.)

CORPUS = [
    # --- Transformers & Attention (sub-topic cluster 1) ---
    "The Transformer architecture uses self-attention mechanisms to weigh the importance of each token "
    "relative to all others in a sequence, allowing parallel computation unlike RNNs.",

    "Multi-head attention splits the embedding space into multiple subspaces so the model can "
    "simultaneously attend to different representation aspects, such as syntax and semantics.",

    "Positional encodings are added to token embeddings in Transformers because self-attention is "
    "permutation-invariant; sine and cosine functions of different frequencies encode each position.",

    # --- Neural Network Training (sub-topic cluster 2) ---
    "Backpropagation computes gradients of the loss with respect to each weight by applying the chain "
    "rule layer by layer, enabling gradient-descent optimizers to update parameters.",

    "Stochastic Gradient Descent (SGD) with momentum accumulates a velocity vector in the direction "
    "of persistent gradient components, helping escape shallow local minima and saddle points.",

    "The Adam optimizer combines the adaptive learning rates of RMSProp with the momentum of SGD, "
    "maintaining per-parameter first and second moment estimates to normalize gradient steps.",

    # --- Regularization & Generalization (sub-topic cluster 3) ---
    "Dropout randomly zeroes neuron activations during training with probability p, forcing the network "
    "to learn redundant representations and reducing co-adaptation between neurons.",

    "Batch Normalization normalizes layer inputs to zero mean and unit variance, then applies learned "
    "scale and shift parameters, stabilizing training and allowing higher learning rates.",

    "L2 regularization (weight decay) adds a penalty proportional to the squared magnitude of weights "
    "to the loss function, discouraging large weights and improving generalization.",

    # --- Keyword / Jargon-heavy documents (BM25 advantage) ---
    "BLEU (Bilingual Evaluation Understudy) score measures n-gram precision overlap between a machine-"
    "translated output and human reference translations, with a brevity penalty for short outputs.",

    "RLHF (Reinforcement Learning from Human Feedback) fine-tunes a pre-trained language model using "
    "a reward model trained on human preference rankings via Proximal Policy Optimization (PPO).",

    "The Mixtral 8x7B model uses a Sparse Mixture-of-Experts (SMoE) architecture, routing each token "
    "through only 2 of 8 expert feed-forward networks, reducing active parameters during inference.",
]

print(f"✅ Corpus loaded: {len(CORPUS)} documents")
for i, doc in enumerate(CORPUS):
    print(f"  [{i:02d}] {doc[:80]}...")

✅ Corpus loaded: 12 documents
  [00] The Transformer architecture uses self-attention mechanisms to weigh the importa...
  [01] Multi-head attention splits the embedding space into multiple subspaces so the m...
  [02] Positional encodings are added to token embeddings in Transformers because self-...
  [03] Backpropagation computes gradients of the loss with respect to each weight by ap...
  [04] Stochastic Gradient Descent (SGD) with momentum accumulates a velocity vector in...
  [05] The Adam optimizer combines the adaptive learning rates of RMSProp with the mome...
  [06] Dropout randomly zeroes neuron activations during training with probability p, f...
  [07] Batch Normalization normalizes layer inputs to zero mean and unit variance, then...
  [08] L2 regularization (weight decay) adds a penalty proportional to the squared magn...
  [09] BLEU (Bilingual Evaluation Understudy) score measures n-gram precision overlap b...
  [10] RLHF (Reinforcement Learning from Human Feedback) fin

In [24]:
# Cell 4: HybridRetriever — BM25 + SBERT + Reciprocal Rank Fusion (RRF)

import numpy as np
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, util

class HybridRetriever:
    """
    Combines BM25 (keyword/lexical) and SBERT (dense/semantic) retrieval
    via Reciprocal Rank Fusion (RRF).

    RRF formula: score(d) = Σ 1 / (k + rank(d))
    where k=60 is a smoothing constant that reduces the impact of high ranks.
    """

    def __init__(self, corpus: list[str], k: int = 60):
        self.corpus = corpus
        self.k = k

        # --- BM25 Setup ---
        # Tokenize on whitespace after lowercasing (BM25 is case-sensitive)
        tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(tokenized)
        print("✅ BM25 index built.")

        # --- SBERT Setup ---
        print("⏳ Loading SBERT model (this may take ~30s on first run)...")
        self.sbert = SentenceTransformer("all-MiniLM-L6-v2")
        # Pre-encode all corpus documents once
        self.corpus_embeddings = self.sbert.encode(
            corpus, convert_to_tensor=True, show_progress_bar=False
        )
        print("✅ SBERT index built.")

    def retrieve(self, query: str, top_k: int = 5) -> list[dict]:
        """
        Run hybrid retrieval and return top_k documents ranked by RRF score.

        Returns list of dicts with keys:
            doc_id, rrf_score, bm25_rank, sbert_rank, text
        """
        n = len(self.corpus)

        # ── BM25 Ranking ──────────────────────────────────────────────────────
        bm25_scores = self.bm25.get_scores(query.lower().split())
        # argsort ascending → reverse for descending rank
        bm25_ranked_ids = np.argsort(bm25_scores)[::-1]
        # Map doc_id → 1-based rank
        bm25_rank_map = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_ranked_ids)}

        # ── SBERT Ranking ─────────────────────────────────────────────────────
        query_embedding = self.sbert.encode(query, convert_to_tensor=True)
        cosine_scores = util.cos_sim(query_embedding, self.corpus_embeddings)[0]
        sbert_ranked_ids = np.argsort(cosine_scores.cpu().numpy())[::-1]
        sbert_rank_map = {doc_id: rank + 1 for rank, doc_id in enumerate(sbert_ranked_ids)}

        # ── Reciprocal Rank Fusion ────────────────────────────────────────────
        rrf_scores = {}
        for doc_id in range(n):
            r_bm25  = bm25_rank_map[doc_id]
            r_sbert = sbert_rank_map[doc_id]
            rrf_scores[doc_id] = (1 / (self.k + r_bm25)) + (1 / (self.k + r_sbert))

        # Sort by RRF score descending
        sorted_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)

        results = []
        for doc_id in sorted_ids[:top_k]:
            results.append({
                "doc_id":     doc_id,
                "rrf_score":  round(rrf_scores[doc_id], 6),
                "bm25_rank":  bm25_rank_map[doc_id],
                "sbert_rank": sbert_rank_map[doc_id],
                "text":       self.corpus[doc_id],
            })
        return results


# ── Smoke Test ────────────────────────────────────────────────────────────────
retriever = HybridRetriever(CORPUS)

test_results = retriever.retrieve("how does attention work in transformers?", top_k=3)
print("\n🔍 Smoke test — 'how does attention work in transformers?'")
print("-" * 70)
for r in test_results:
    print(f"  doc_id={r['doc_id']} | RRF={r['rrf_score']} | "
          f"BM25_rank={r['bm25_rank']} | SBERT_rank={r['sbert_rank']}")
    print(f"  ↳ {r['text'][:90]}...\n")

✅ BM25 index built.
⏳ Loading SBERT model (this may take ~30s on first run)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ SBERT index built.

🔍 Smoke test — 'how does attention work in transformers?'
----------------------------------------------------------------------
  doc_id=2 | RRF=0.032522 | BM25_rank=2 | SBERT_rank=1
  ↳ Positional encodings are added to token embeddings in Transformers because self-attention ...

  doc_id=1 | RRF=0.032266 | BM25_rank=1 | SBERT_rank=3
  ↳ Multi-head attention splits the embedding space into multiple subspaces so the model can s...

  doc_id=0 | RRF=0.032002 | BM25_rank=3 | SBERT_rank=2
  ↳ The Transformer architecture uses self-attention mechanisms to weigh the importance of eac...



In [25]:
# Cell 5: Cross-Encoder Re-Ranker using ms-marco-MiniLM-L-6-v2

from sentence_transformers import CrossEncoder

print("⏳ Loading cross-encoder (this may take ~30s)...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("✅ Cross-encoder loaded.")

def rerank(query: str, candidates: list[dict], top_k: int = 3) -> list[dict]:
    """
    Re-rank candidate documents using a cross-encoder.

    Args:
        query      : The ORIGINAL user query (not the expanded version).
        candidates : List of dicts from HybridRetriever.retrieve()
        top_k      : Number of top documents to return after re-ranking.

    Returns:
        List of candidate dicts, augmented with 'ce_score', sorted by ce_score desc.

    Note: Cross-encoder scores can be negative — higher (less negative) = more relevant.
    """
    if not candidates:
        return []

    # Build (query, document) pairs for the cross-encoder
    pairs = [(query, c["text"]) for c in candidates]

    # Score all pairs in one batched forward pass
    ce_scores = cross_encoder.predict(pairs)

    # Attach scores to candidates
    for candidate, score in zip(candidates, ce_scores):
        candidate["ce_score"] = round(float(score), 4)

    # Sort by cross-encoder score descending and return top_k
    reranked = sorted(candidates, key=lambda x: x["ce_score"], reverse=True)
    return reranked[:top_k]


# ── Smoke Test ────────────────────────────────────────────────────────────────
test_candidates = retriever.retrieve("attention mechanism transformers", top_k=6)
reranked = rerank("how does attention work in transformers?", test_candidates, top_k=3)

print("\n🎯 Smoke test — Cross-Encoder Re-Ranking")
print("-" * 70)
for i, r in enumerate(reranked):
    print(f"  Rank {i+1} | CE Score: {r['ce_score']:>8.4f} | "
          f"RRF rank was #{sorted(test_candidates, key=lambda x: x['rrf_score'], reverse=True).index(r)+1}")
    print(f"  ↳ {r['text'][:90]}...\n")

⏳ Loading cross-encoder (this may take ~30s)...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Cross-encoder loaded.

🎯 Smoke test — Cross-Encoder Re-Ranking
----------------------------------------------------------------------
  Rank 1 | CE Score:   4.2254 | RRF rank was #1
  ↳ Positional encodings are added to token embeddings in Transformers because self-attention ...

  Rank 2 | CE Score:   1.3734 | RRF rank was #5
  ↳ The Transformer architecture uses self-attention mechanisms to weigh the importance of eac...

  Rank 3 | CE Score:  -1.7791 | RRF rank was #2
  ↳ Multi-head attention splits the embedding space into multiple subspaces so the model can s...



In [26]:
# Cell 6: Query Expansion — HyDE (Hypothetical Document Embeddings) using Gemini

import google.generativeai as genai

genai.configure(api_key=GEMINI_API_KEY)

# Use temperature=0.0 for deterministic, factual hypothetical documents
gemini_model = genai.GenerativeModel(
    model_name="gemini-2.5-flash",
    generation_config=genai.GenerationConfig(temperature=0.0, max_output_tokens=200),
)

def hyde_expand(user_query: str) -> str:
    """
    HyDE (Hypothetical Document Embeddings):
    Ask the LLM to write a short hypothetical answer/document as if it were
    a passage from a textbook. Use THAT text as the retrieval query instead
    of the raw user question.

    Why it works: The hypothetical document uses technical vocabulary
    that bridges the gap between a vague student question and precise corpus docs.
    """
    prompt = (
        "You are a technical AI/ML textbook author. "
        "Write a concise 2-3 sentence passage that would appear in a textbook "
        "and directly answers the following question. "
        "Use precise technical terminology. Do NOT say 'This passage...' — just write the content.\n\n"
        f"Question: {user_query}"
    )
    response = gemini_model.generate_content(prompt)
    hypothetical_doc = response.text.strip()
    return hypothetical_doc


# ── Smoke Test ────────────────────────────────────────────────────────────────
test_query = "what is attention?"
expanded = hyde_expand(test_query)

print(f"📝 Original query : '{test_query}'")
print(f"💡 HyDE expansion :\n{expanded}")

📝 Original query : 'what is attention?'
💡 HyDE expansion :
Attention is a mechanism that allows a


In [27]:
# Cell 7: End-to-End Advanced RAG Pipeline

from groq import Groq

groq_client = Groq(api_key=GROQ_API_KEY)

def generate_answer(user_query: str, context_docs: list[dict]) -> str:
    """
    Generate a final answer using Groq (llama-3.1-8b-instant) with the
    top re-ranked documents as grounding context.
    """
    context_text = "\n\n".join(
        [f"[Doc {i+1}]: {doc['text']}" for i, doc in enumerate(context_docs)]
    )
    prompt = (
        f"You are a helpful university AI/ML tutor. "
        f"Answer the student's question using ONLY the context provided below. "
        f"Be concise and clear. If the context doesn't contain the answer, say so.\n\n"
        f"CONTEXT:\n{context_text}\n\n"
        f"STUDENT QUESTION: {user_query}\n\n"
        f"ANSWER:"
    )
    response = groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3,
        max_tokens=300,
    )
    return response.choices[0].message.content.strip()


def advanced_rag(user_query: str, verbose: bool = False) -> str:
    """
    Full Advanced RAG Pipeline:
        1. Query Expansion  → HyDE generates a hypothetical answer
        2. Hybrid Retrieval → BM25 + SBERT + RRF on the expanded query
        3. Re-Ranking       → Cross-encoder re-ranks using the ORIGINAL query
        4. LLM Generation   → Groq generates the final grounded answer

    Args:
        user_query : The raw student question.
        verbose    : If True, prints intermediate pipeline steps.

    Returns:
        Final answer string.
    """
    if verbose:
        print(f"{'='*65}")
        print(f"🟦 ADVANCED RAG | Query: '{user_query}'")
        print(f"{'='*65}")

    # ── Step 1: Query Expansion (HyDE) ───────────────────────────────────────
    expanded_query = hyde_expand(user_query)
    if verbose:
        print(f"\n💡 [Step 1] HyDE Expansion:\n   {expanded_query[:120]}...")

    # ── Step 2: Hybrid Retrieval (on expanded query) ──────────────────────────
    candidates = retriever.retrieve(expanded_query, top_k=6)
    if verbose:
        print(f"\n🔍 [Step 2] Hybrid Retrieval — Top {len(candidates)} candidates:")
        for c in candidates:
            print(f"   RRF={c['rrf_score']} | BM25_rank={c['bm25_rank']} | "
                  f"SBERT_rank={c['sbert_rank']} | {c['text'][:60]}...")

    # ── Step 3: Cross-Encoder Re-Ranking (on ORIGINAL query) ─────────────────
    top_docs = rerank(user_query, candidates, top_k=3)  # ← original query!
    if verbose:
        print(f"\n🎯 [Step 3] Re-Ranked Top 3 (by Cross-Encoder):")
        for i, doc in enumerate(top_docs):
            print(f"   [{i+1}] CE={doc['ce_score']:>8.4f} | {doc['text'][:70]}...")

    # ── Step 4: LLM Generation ────────────────────────────────────────────────
    answer = generate_answer(user_query, top_docs)
    if verbose:
        print(f"\n🤖 [Step 4] Generated Answer:\n   {answer}")
        print(f"{'='*65}\n")

    return answer


# ── Smoke Test ────────────────────────────────────────────────────────────────
test_answer = advanced_rag("what is attention?", verbose=True)

🟦 ADVANCED RAG | Query: 'what is attention?'

💡 [Step 1] HyDE Expansion:
   Attention is a mechanism that allows a...

🔍 [Step 2] Hybrid Retrieval — Top 6 candidates:
   RRF=0.032522 | BM25_rank=2 | SBERT_rank=1 | Multi-head attention splits the embedding space into multipl...
   RRF=0.032266 | BM25_rank=1 | SBERT_rank=3 | Positional encodings are added to token embeddings in Transf...
   RRF=0.03125 | BM25_rank=4 | SBERT_rank=4 | RLHF (Reinforcement Learning from Human Feedback) fine-tunes...
   RRF=0.03009 | BM25_rank=8 | SBERT_rank=5 | Dropout randomly zeroes neuron activations during training w...
   RRF=0.030018 | BM25_rank=12 | SBERT_rank=2 | The Transformer architecture uses self-attention mechanisms ...
   RRF=0.029877 | BM25_rank=5 | SBERT_rank=9 | L2 regularization (weight decay) adds a penalty proportional...

🎯 [Step 3] Re-Ranked Top 3 (by Cross-Encoder):
   [1] CE= -1.0534 | Multi-head attention splits the embedding space into multiple subspace...
   [2] CE= -3.7536 | Posi

In [28]:
# Cell 8: Naïve RAG Baseline
# Dense-only (SBERT cosine similarity), no query expansion, no re-ranking

def naive_rag(user_query: str, top_k: int = 3, verbose: bool = False) -> str:
    """
    Naïve RAG Baseline:
        1. Encode query with SBERT
        2. Cosine similarity against corpus embeddings
        3. Take top_k directly (no re-ranking)
        4. Generate answer with Groq

    This is the baseline to compare against Advanced RAG.
    """
    if verbose:
        print(f"{'='*65}")
        print(f"🟥 NAÏVE RAG | Query: '{user_query}'")
        print(f"{'='*65}")

    # Dense-only retrieval
    query_emb   = retriever.sbert.encode(user_query, convert_to_tensor=True)
    cos_scores  = util.cos_sim(query_emb, retriever.corpus_embeddings)[0].cpu().numpy()
    top_indices = np.argsort(cos_scores)[::-1][:top_k]

    top_docs = [
        {"text": CORPUS[i], "cosine_score": round(float(cos_scores[i]), 4)}
        for i in top_indices
    ]

    if verbose:
        print(f"\n🔍 Top {top_k} by cosine similarity:")
        for i, doc in enumerate(top_docs):
            print(f"   [{i+1}] score={doc['cosine_score']:.4f} | {doc['text'][:70]}...")

    answer = generate_answer(user_query, top_docs)

    if verbose:
        print(f"\n🤖 Generated Answer:\n   {answer}")
        print(f"{'='*65}\n")

    return answer


# ── Smoke Test ────────────────────────────────────────────────────────────────
_ = naive_rag("what is attention?", verbose=True)

🟥 NAÏVE RAG | Query: 'what is attention?'

🔍 Top 3 by cosine similarity:
   [1] score=0.4919 | Multi-head attention splits the embedding space into multiple subspace...
   [2] score=0.3728 | The Transformer architecture uses self-attention mechanisms to weigh t...
   [3] score=0.2610 | Positional encodings are added to token embeddings in Transformers bec...

🤖 Generated Answer:
   Attention refers to the mechanism used in the Transformer architecture to weigh the importance of each token relative to all others in a sequence. This allows the model to focus on different aspects of the input, such as syntax and semantics, simultaneously.



In [29]:
# Cell 9: Run the Comparison Experiment
# 3 queries through both pipelines — capture top docs and answers

import textwrap

def get_top_doc_naive(query: str) -> str:
    """Return the top document text from Naïve RAG."""
    query_emb  = retriever.sbert.encode(query, convert_to_tensor=True)
    cos_scores = util.cos_sim(query_emb, retriever.corpus_embeddings)[0].cpu().numpy()
    top_idx    = int(np.argmax(cos_scores))
    return CORPUS[top_idx]

def get_top_doc_advanced(query: str) -> str:
    """Return the top document text after full Advanced RAG pipeline."""
    expanded   = hyde_expand(query)
    candidates = retriever.retrieve(expanded, top_k=6)
    reranked   = rerank(query, candidates, top_k=1)
    return reranked[0]["text"]


# ── Define the 3 Test Queries ─────────────────────────────────────────────────
TEST_QUERIES = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is RLHF and how does it use human feedback?",   # ← your own query
]

# ── Run Both Pipelines ────────────────────────────────────────────────────────
print("⏳ Running comparison experiment (this takes ~1 minute)...\n")

results = []
for query in TEST_QUERIES:
    print(f"  Processing: '{query}'")
    naive_top    = get_top_doc_naive(query)
    advanced_top = get_top_doc_advanced(query)
    naive_ans    = naive_rag(query)
    advanced_ans = advanced_rag(query)
    different    = naive_top.strip() != advanced_top.strip()

    results.append({
        "query":        query,
        "naive_top":    naive_top,
        "advanced_top": advanced_top,
        "naive_ans":    naive_ans,
        "advanced_ans": advanced_ans,
        "different":    different,
    })

print("\n✅ Done!\n")

# ── Pretty-print Results ──────────────────────────────────────────────────────
wrap = lambda t: "\n           ".join(textwrap.wrap(t, width=70))

for i, r in enumerate(results):
    print(f"{'─'*70}")
    print(f"Query {i+1}: \"{r['query']}\"")
    print(f"\n  [Naïve Top Doc]")
    print(f"           {wrap(r['naive_top'])}")
    print(f"\n  [Advanced Top Doc]")
    print(f"           {wrap(r['advanced_top'])}")
    print(f"\n  ➤ Different top doc? {'YES ✅' if r['different'] else 'NO ❌'}")
    print(f"\n  [Naïve Answer]    {r['naive_ans'][:120]}...")
    print(f"  [Advanced Answer] {r['advanced_ans'][:120]}...")
    print()

⏳ Running comparison experiment (this takes ~1 minute)...

  Processing: 'how do transformers encode meaning?'
  Processing: 'optimization techniques for training'
  Processing: 'what is RLHF and how does it use human feedback?'

✅ Done!

──────────────────────────────────────────────────────────────────────
Query 1: "how do transformers encode meaning?"

  [Naïve Top Doc]
           Positional encodings are added to token embeddings in Transformers
           because self-attention is permutation-invariant; sine and cosine
           functions of different frequencies encode each position.

  [Advanced Top Doc]
           Positional encodings are added to token embeddings in Transformers
           because self-attention is permutation-invariant; sine and cosine
           functions of different frequencies encode each position.

  ➤ Different top doc? NO ❌

  [Naïve Answer]    Transformers encode meaning through self-attention mechanisms, which weigh the importance of each token rela

In [30]:
# Cell 10: Print the Markdown comparison table (copy output into a Markdown cell)

def shorten(text: str, n: int = 55) -> str:
    return text[:n].replace("|", "—") + "…"

header = (
    "| # | Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Different? |\n"
    "|---|-------|-------------------|----------------------|------------|\n"
)

rows = ""
for i, r in enumerate(results):
    rows += (
        f"| {i+1} | {r['query']} "
        f"| {shorten(r['naive_top'])} "
        f"| {shorten(r['advanced_top'])} "
        f"| {'✅ Yes' if r['different'] else '❌ No'} |\n"
    )

print("```markdown")
print(header + rows)
print("```")
print("\n📋 Paste the above table into a new Markdown cell in your notebook.")

```markdown
| # | Query | Naïve RAG Top Doc | Advanced RAG Top Doc | Different? |
|---|-------|-------------------|----------------------|------------|
| 1 | how do transformers encode meaning? | Positional encodings are added to token embeddings in T… | Positional encodings are added to token embeddings in T… | ❌ No |
| 2 | optimization techniques for training | The Adam optimizer combines the adaptive learning rates… | RLHF (Reinforcement Learning from Human Feedback) fine-… | ✅ Yes |
| 3 | what is RLHF and how does it use human feedback? | RLHF (Reinforcement Learning from Human Feedback) fine-… | RLHF (Reinforcement Learning from Human Feedback) fine-… | ❌ No |

```

📋 Paste the above table into a new Markdown cell in your notebook.


In [31]:
# Cell 11 (BONUS): Weighted RRF — experiment with alpha weighting BM25 vs SBERT

def weighted_rrf_retrieve(query: str, alpha: float = 0.5, top_k: int = 5) -> list[dict]:
    """
    Weighted RRF:
        score(d) = α * 1/(k + r_BM25) + (1-α) * 1/(k + r_SBERT)

    alpha=1.0  → pure BM25  (good for keyword/jargon queries)
    alpha=0.0  → pure SBERT (good for semantic/paraphrase queries)
    alpha=0.5  → balanced   (default RRF)
    """
    k = retriever.k
    n = len(CORPUS)

    # BM25 ranks
    bm25_scores    = retriever.bm25.get_scores(query.lower().split())
    bm25_ranked    = np.argsort(bm25_scores)[::-1]
    bm25_rank_map  = {doc_id: rank + 1 for rank, doc_id in enumerate(bm25_ranked)}

    # SBERT ranks
    q_emb          = retriever.sbert.encode(query, convert_to_tensor=True)
    cos_scores     = util.cos_sim(q_emb, retriever.corpus_embeddings)[0].cpu().numpy()
    sbert_ranked   = np.argsort(cos_scores)[::-1]
    sbert_rank_map = {doc_id: rank + 1 for rank, doc_id in enumerate(sbert_ranked)}

    # Weighted RRF
    rrf_scores = {
        doc_id: alpha * (1 / (k + bm25_rank_map[doc_id]))
                + (1 - alpha) * (1 / (k + sbert_rank_map[doc_id]))
        for doc_id in range(n)
    }

    sorted_ids = sorted(rrf_scores, key=rrf_scores.get, reverse=True)
    return [
        {
            "doc_id": did,
            "rrf_score": round(rrf_scores[did], 6),
            "bm25_rank": bm25_rank_map[did],
            "sbert_rank": sbert_rank_map[did],
            "text": CORPUS[did],
        }
        for did in sorted_ids[:top_k]
    ]


# ── Alpha Experiment ──────────────────────────────────────────────────────────
# Keyword-heavy query → higher alpha (more BM25) should help
# Semantic query      → lower alpha (more SBERT) should help

experiments = [
    ("BLEU score machine translation",       "keyword-heavy → expect alpha=0.7 best"),
    ("how do models learn representations?", "semantic      → expect alpha=0.3 best"),
]

for query, note in experiments:
    print(f"\nQuery: '{query}'  [{note}]")
    print(f"{'alpha':>8} | {'Top Doc':60}")
    print("-" * 75)
    for alpha in [0.3, 0.5, 0.7]:
        top = weighted_rrf_retrieve(query, alpha=alpha, top_k=1)[0]
        print(f"  α={alpha}  | {top['text'][:65]}...")


Query: 'BLEU score machine translation'  [keyword-heavy → expect alpha=0.7 best]
   alpha | Top Doc                                                     
---------------------------------------------------------------------------
  α=0.3  | BLEU (Bilingual Evaluation Understudy) score measures n-gram prec...
  α=0.5  | BLEU (Bilingual Evaluation Understudy) score measures n-gram prec...
  α=0.7  | BLEU (Bilingual Evaluation Understudy) score measures n-gram prec...

Query: 'how do models learn representations?'  [semantic      → expect alpha=0.3 best]
   alpha | Top Doc                                                     
---------------------------------------------------------------------------
  α=0.3  | Dropout randomly zeroes neuron activations during training with p...
  α=0.5  | Dropout randomly zeroes neuron activations during training with p...
  α=0.7  | Dropout randomly zeroes neuron activations during training with p...
